In [ ]:
# A default setup cell.
# It imports environment variables, define 'devtools.debug" as a buildins, set PYTHONPATH, and code auto-reload
# Copy it in other Notebooks

%load_ext autoreload
%autoreload 2
%reset -f

from devtools import debug  # noqa: F401  # noqa: F811
from dotenv import load_dotenv
from rich import print  # noqa: F401

assert load_dotenv(verbose=True)

In [ ]:
from src.utils.config_mngr import global_config, global_config_reload

global_config_reload()

list_demos = global_config().merge_with("config/demos/document_extractor.yaml").get_list("Document_extractor_demo")


test_schema = next((item for item in list_demos if item.get("schema_name") == "test"))


In [ ]:
from src.demos.ekg.pydantic_rag import PydanticRag

url = global_config().get_dsn("vector_store.postgres_url", driver="asyncpg")

rag = PydanticRag(
    model_definition=test_schema, postgres_url=url, embeddings_id="qwen3_06b_deepinfra", llm_id=None, kv_store_id="file"
)


In [ ]:
debug(rag)

In [ ]:
a = rag.get_key()
a

In [ ]:
# A tiny markdown document

doc_text = """
# Jane Doe
Jane is 29 years old and can be reached at jane.doe@example.com (personal adress) and J.doe@apple.com (professional email).
She lives rue de la Concorde, 31000 Toulouse France.

"""
doc_id = "test"
key = "example Jane Joe #1"
# 1. Analyse → structured Pydantic object

person = rag.analyze_document(
    document_id=doc_id,
    markdown=doc_text,
)

print("Structured result:", person)

assert person


In [ ]:
chunks = rag.chunck(person)


In [ ]:
debug(chunks)

In [26]:
debug(person)

/tmp/ipykernel_377137/3079920801.py:1 <module>
    person: Person(
        name='Jane Doe',
        age=29,
        email=[
            Email(
                url='jane.doe@example.com',
                email_type='personal',
            ),
            Email(
                url='J.doe@apple.com',
                email_type='professional',
            ),
        ],
        address=Address(
            street='rue de la Concorde',
            city='Toulouse',
            zip_code='31000',
            country='France',
        ),
        doc_id='test',
    ) (Person)


Person(name='Jane Doe', age=29, email=[Email(url='jane.doe@example.com', email_type='personal'), Email(url='J.doe@apple.com', email_type='professional')], address=Address(street='rue de la Concorde', city='Toulouse', zip_code='31000', country='France'), doc_id='test')

In [32]:
person.name

'Jane Doe'

In [33]:
key_path = rag._key_field.split(".")
print(key_path)
model_data = rag.model_dump()

#debug(model_data)

a = person.__getattribute__('name')
a

['Person', 'name']

'Jane Doe'

In [ ]:
# 2. Index the document
#rag.store_chunks(chunks)
print("Document stored.")



In [ ]:
hits = rag.query_vectorstore("e-mail address", k=2)
print("Vector hits:", hits)

In [ ]:
# 3. Query the vector store
hits = rag.query_vectorstore("e-mail address", k=2, filter={"langchain_metadata ->> description": {"$eq":"Age in years"}})
print("Vector hits:", hits)